# CP5 · Examen blanc de la partie V — vérification de ta copie

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/checkpoints/partie_5/02_examen_notebook.ipynb)

L'examen se fait **sur papier** (`01_examen_sujet.md`, 112 minutes, fiches fermées). Ce notebook ne sert qu'**après** : il vérifie les réponses chiffrées de ta copie. Ses cellules ne vérifient rien tant que `EXAM_OVER` vaut `False`, et « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli (⏸️).

> Travaille dans **ta copie** (`mon_travail/checkpoints/partie_5/02_examen_notebook.ipynb`, créée par `python tools/start_chapter.py CP5`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

## Après l'examen : vérification automatique

**Ne commence qu'une fois ta copie terminée**, examen rendu. Ce notebook vérifie les réponses que tu as écrites sur ta feuille, question par question : ✅ juste, ❌ faux (avec une piste), ⏳ pas rempli. Il ne note pas : les démarches, les justifications, le quiz CP5.Q1, la démonstration CP5.8 et les corrections de CP5.9 b) se corrigent avec `03_examen_corrige.md`, qui donne aussi le barème.

1. Passe `EXAM_OVER` à `True` dans la cellule ci-dessous, puis exécute-la.
2. Reporte chaque réponse de ta feuille : **la valeur** que tu as écrite, pas un nouveau calcul (sinon tu ne vérifies plus ta copie). Laisse `...` pour une question que tu n'as pas faite.
3. Si une cellule s'arrête sur une erreur, « Tout exécuter » s'arrête avec elle : corrige-la, puis relance cette cellule et les suivantes, jusqu'au bilan.
4. Les formes d'écriture : en Python, le séparateur décimal est un **point** (`2.75`) : une virgule arrête toute la cellule (`0,027`), et aucune réponse de la question n'est vérifiée, ou donne un couple de nombres (`0,75`) ; une liste s'écrit entre crochets (`[-1.5, 4]`), une matrice comme la liste de ses lignes (`[[1, -2], [0.5, 3]]`), un nombre complexe $a + ib$ comme la liste `[a, b]` (ou `a + bj`, à la manière de Python) ; un texte entre guillemets (`"XYZ"`), comme une fraction (`"3/8"`) ou un angle exact (`"pi/5"`) ; un code de comportement (1, 2 ou 3) s'écrit sans guillemets.
5. Le bilan, à la fin, dit aussi quelles réponses n'ont pas été vérifiées.

In [ ]:
EXAM_OVER = False   # set it to True once your exam is finished, then run the cells below

RESULTS = {}        # sub-ID -> result of wbm.check
_REMINDED = []      # the full reminder is printed once, then a short one


def exam_over():
    """True once the exam is finished; otherwise print a reminder (and the cell checks nothing)."""
    if not EXAM_OVER:
        print("⏸️ Rien n'est vérifié (EXAM_OVER vaut False)." if _REMINDED else
              "⏸️ Vérification après l'examen : passe EXAM_OVER à True dans cette cellule, puis exécute-la.")
        _REMINDED.append(True)
        return False
    if "wbm" not in globals():
        print("⚠️ Exécute d'abord la cellule de configuration, tout en haut du notebook.")
        return False
    return True


def as_written(answer):
    """A complex number typed as such (0.5 + 2j) is read as the list [a, b] that the statement asks for."""
    if isinstance(answer, complex):
        return [answer.real, answer.imag]
    return answer

**CP5.1 — Convolution de deux suites, corrélation croisée et matrice de Toeplitz**

In [ ]:
# CP5.1: the values written on your answer sheet
answer_CP5_1a = ...  # a) a list of 6 integers
answer_CP5_1b = ...  # b) an integer
answer_CP5_1c = ...  # c) a list of 3 integers
answer_CP5_1d = ...  # d) a list of 9 integers
answer_CP5_1e = ...  # e) a list of 6 integers
answer_CP5_1f = ...  # f) a 6 × 3 matrix, as a list of rows
answer_CP5_1g = ...  # g) an integer
answer_CP5_1h = ...  # h) a list of 6 integers

if exam_over():
    for letter, answer in zip("abcdefgh", [answer_CP5_1a, answer_CP5_1b, answer_CP5_1c, answer_CP5_1d, answer_CP5_1e, answer_CP5_1f, answer_CP5_1g, answer_CP5_1h]):
        RESULTS[f"CP5.1{letter}"] = wbm.check(f"CP5.1{letter}", as_written(answer))

**CP5.2 — Positions d'une fenêtre, pas, dilatation, et la convolution transposée d'un signal**

In [ ]:
# CP5.2: the values written on your answer sheet
answer_CP5_2a = ...  # a) an integer
answer_CP5_2b = ...  # b) an integer
answer_CP5_2c = ...  # c) an integer
answer_CP5_2d = ...  # d) the window lengths k, as a list of integers
answer_CP5_2e = ...  # e) a list of 3 integers
answer_CP5_2f = ...  # f) a list of 9 integers
answer_CP5_2g = ...  # g) an integer
answer_CP5_2h = ...  # h) the different values, as a list of integers

if exam_over():
    for letter, answer in zip("abcdefgh", [answer_CP5_2a, answer_CP5_2b, answer_CP5_2c, answer_CP5_2d, answer_CP5_2e, answer_CP5_2f, answer_CP5_2g, answer_CP5_2h]):
        RESULTS[f"CP5.2{letter}"] = wbm.check(f"CP5.2{letter}", as_written(answer))

**CP5.3 — Une grille filtrée par un noyau séparable : paramètres, coût et champ récepteur**

In [ ]:
# CP5.3: the values written on your answer sheet
answer_CP5_3a = ...  # a) [a, b], a list of 2 lists
answer_CP5_3b = ...  # b) a 2 × 3 matrix, as a list of rows
answer_CP5_3c = ...  # c) a list of 2 integers
answer_CP5_3d = ...  # d) a list of 3 integers
answer_CP5_3e = ...  # e) an integer
answer_CP5_3f = ...  # f) an integer
answer_CP5_3g = ...  # g) an integer
answer_CP5_3h = ...  # h) a list of 2 integers

if exam_over():
    for letter, answer in zip("abcdefgh", [answer_CP5_3a, answer_CP5_3b, answer_CP5_3c, answer_CP5_3d, answer_CP5_3e, answer_CP5_3f, answer_CP5_3g, answer_CP5_3h]):
        RESULTS[f"CP5.3{letter}"] = wbm.check(f"CP5.3{letter}", as_written(answer))

**CP5.4 — Équivariance, matrice circulante et le pire cas d'une perturbation bornée**

In [ ]:
# CP5.4: the values written on your answer sheet
answer_CP5_4a = ...  # a) a list of 5 integers
answer_CP5_4b = ...  # b) a list of 5 integers
answer_CP5_4c = ...  # c) a list of 5 integers
answer_CP5_4d = ...  # d) a list of 5 numbers
answer_CP5_4e = ...  # e) a number
answer_CP5_4f = ...  # f) a number (3 decimals)

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP5_4a, answer_CP5_4b, answer_CP5_4c, answer_CP5_4d, answer_CP5_4e, answer_CP5_4f]):
        RESULTS[f"CP5.4{letter}"] = wbm.check(f"CP5.4{letter}", as_written(answer))

**CP5.5 — Une récurrence avec entrée : solution et réponse impulsionnelle**

In [ ]:
# CP5.5: the values written on your answer sheet
answer_CP5_5a = ...  # a) a list of 2 numbers
answer_CP5_5b = ...  # b) a list of 2 numbers
answer_CP5_5c = ...  # c) a list of 2 numbers, in increasing order
answer_CP5_5d = ...  # d) a list of 4 numbers
answer_CP5_5e = ...  # e) a number (4 decimals)
answer_CP5_5f = ...  # f) a number
answer_CP5_5g = ...  # g) a list of 2 numbers
answer_CP5_5h = ...  # h) a number

if exam_over():
    for letter, answer in zip("abcdefgh", [answer_CP5_5a, answer_CP5_5b, answer_CP5_5c, answer_CP5_5d, answer_CP5_5e, answer_CP5_5f, answer_CP5_5g, answer_CP5_5h]):
        RESULTS[f"CP5.5{letter}"] = wbm.check(f"CP5.5{letter}", as_written(answer))

**CP5.6 — Valeurs propres complexes, rayon spectral, normes et croissance transitoire**

In [ ]:
# CP5.6: the values written on your answer sheet
answer_CP5_6a = ...  # a) a list [a, b] for a + ib (4 decimals)
answer_CP5_6b = ...  # b) a number (4 decimals)
answer_CP5_6c = ...  # c) an exact angle in radians, as text (like "pi/5")
answer_CP5_6d = ...  # d) a matrix, as a list of rows
answer_CP5_6e = ...  # e) a matrix, as a list of rows
answer_CP5_6f = ...  # f) a list of 2 numbers
answer_CP5_6g = ...  # g) a number (4 decimals)
answer_CP5_6h = ...  # h) a list of 2 codes (1, 2 or 3)

if exam_over():
    for letter, answer in zip("abcdefgh", [answer_CP5_6a, answer_CP5_6b, answer_CP5_6c, answer_CP5_6d, answer_CP5_6e, answer_CP5_6f, answer_CP5_6g, answer_CP5_6h]):
        RESULTS[f"CP5.6{letter}"] = wbm.check(f"CP5.6{letter}", as_written(answer))

**CP5.7 — Produits de jacobiennes le long d'une trajectoire, et sensibilité à un paramètre utilisé à chaque pas**

In [ ]:
# CP5.7: the values written on your answer sheet
answer_CP5_7a = ...  # a) a list of 2 numbers
answer_CP5_7b = ...  # b) a matrix, as a list of rows
answer_CP5_7c = ...  # c) a list of 2 numbers
answer_CP5_7d = ...  # d) a list of 3 numbers
answer_CP5_7e = ...  # e) a number
answer_CP5_7f = ...  # f) a number

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP5_7a, answer_CP5_7b, answer_CP5_7c, answer_CP5_7d, answer_CP5_7e, answer_CP5_7f]):
        RESULTS[f"CP5.7{letter}"] = wbm.check(f"CP5.7{letter}", as_written(answer))

**CP5.9 — Corriger une copie : cinq erreurs (les lettres seulement)**

In [ ]:
# CP5.9: the values written on your answer sheet
answer_CP5_9a = ...  # a) the letters, as text (like "ABCDE")

if exam_over():
    for letter, answer in zip("a", [answer_CP5_9a]):
        RESULTS[f"CP5.9{letter}"] = wbm.check(f"CP5.9{letter}", as_written(answer))

**CP5.10 — Problème de synthèse : un filtre récursif et un filtre à réponse finie, du signal à la stabilité**

In [ ]:
# CP5.10: the values written on your answer sheet
answer_CP5_10a = ...  # a) a list of 5 numbers
answer_CP5_10b = ...  # b) a list of 5 numbers
answer_CP5_10c = ...  # c) a number (4 decimals)
answer_CP5_10d = ...  # d) a list of 3 numbers
answer_CP5_10e = ...  # e) an integer
answer_CP5_10f = ...  # f) a number
answer_CP5_10g = ...  # g) an irreducible fraction, as text (like "3/8")

if exam_over():
    for letter, answer in zip("abcdefg", [answer_CP5_10a, answer_CP5_10b, answer_CP5_10c, answer_CP5_10d, answer_CP5_10e, answer_CP5_10f, answer_CP5_10g]):
        RESULTS[f"CP5.10{letter}"] = wbm.check(f"CP5.10{letter}", as_written(answer))

## Bilan

In [ ]:
# Summary of the automatic checks
import re

EXPECTED = ['CP5.1a', 'CP5.1b', 'CP5.1c', 'CP5.1d', 'CP5.1e', 'CP5.1f', 'CP5.1g', 'CP5.1h', 'CP5.2a', 'CP5.2b', 'CP5.2c', 'CP5.2d', 'CP5.2e', 'CP5.2f', 'CP5.2g', 'CP5.2h', 'CP5.3a', 'CP5.3b', 'CP5.3c', 'CP5.3d', 'CP5.3e', 'CP5.3f', 'CP5.3g', 'CP5.3h', 'CP5.4a', 'CP5.4b', 'CP5.4c', 'CP5.4d', 'CP5.4e', 'CP5.4f', 'CP5.5a', 'CP5.5b', 'CP5.5c', 'CP5.5d', 'CP5.5e', 'CP5.5f', 'CP5.5g', 'CP5.5h', 'CP5.6a', 'CP5.6b', 'CP5.6c', 'CP5.6d', 'CP5.6e', 'CP5.6f', 'CP5.6g', 'CP5.6h', 'CP5.7a', 'CP5.7b', 'CP5.7c', 'CP5.7d', 'CP5.7e', 'CP5.7f', 'CP5.9a', 'CP5.10a', 'CP5.10b', 'CP5.10c', 'CP5.10d', 'CP5.10e', 'CP5.10f', 'CP5.10g']


def natural(sub_id):
    """CP5.2a before CP5.10a: the question number as a number, then the letter."""
    number, letter = re.match(r"CP5\.(\d+)(.*)", sub_id).groups()
    return int(number), letter


if exam_over():
    done = {k: r for k, r in RESULTS.items() if r.status != "pending"}
    right = sorted((k for k, r in done.items() if r), key=natural)
    wrong = sorted((k for k, r in done.items() if not r), key=natural)
    pending = sorted((k for k, r in RESULTS.items() if r.status == "pending"), key=natural)
    missing = [k for k in EXPECTED if k not in RESULTS]
    print(f"Réponses vérifiées : {len(right)} juste(s) sur {len(done)} ; pas remplies : {len(pending)}.")
    if wrong:
        print("À revoir avec le corrigé :", ", ".join(wrong))
    if missing:
        print("⚠️ Pas vérifiées (cellule pas exécutée, ou arrêtée par une erreur, par exemple une virgule décimale) :",
              ", ".join(missing))
    print("Note ensuite ta copie avec le barème de 03_examen_corrige.md : il donne aussi les points des démarches, "
          "des justifications et des questions rédigées, que cette vérification ne voit pas.")

**Ce que cette vérification ne voit pas** : les démarches et les justifications ; le quiz CP5.Q1 ; la démonstration CP5.8 ; les corrections de CP5.9 b) (seules les lettres de a) sont vérifiées). Tout cela se corrige avec `03_examen_corrige.md`. Remplis ensuite le tableau de ta note, à la fin de ta copie, puis fais la remédiation du corrigé.